[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h10_numbers_invented.ipynb)

# Every number, and why it had to be invented

Each new kind of number answered a question the old ones couldn't: how to write nothing, how to share, how to owe,
how long a diagonal is, what squares to minus one, how to turn things in space. Here you can try the pieces
yourself: watch the remainders of a long division until one comes back, turn a repeating decimal back into a
fraction, hunt for a fraction whose square is exactly 2, turn arrows by multiplying by i, and multiply Hamilton's
quaternions in both orders.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider or a box appears, change
it and watch. Nothing can break: if something looks odd, run everything again from the top. It works in your
browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
from fractions import Fraction
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Text, Dropdown, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

def long_division(n, d, steps):
    """The digits and remainders of n ÷ d after the point, the way you learnt at school."""
    digits, rems, r = [], [], n % d
    for _ in range(steps):
        rems.append(r); r *= 10; digits.append(r // d); r %= d
    return digits, rems

print("Ready. Run the next cell.")

## 1 · Watch the remainders

Type a fraction, like 1/7 or 3/17, and press Enter. Dividing by d, a remainder can only be 0, 1, 2, … up to d − 1, so sooner or later one comes back,
and from then on the digits repeat. A remainder of 0 means the decimal ends. The coral box is the first remainder
that comes back.

In [ ]:
def remainders(fraction="1/7"):
    try:
        top, bottom = (int(x) for x in fraction.replace(" ", "").split("/"))
    except ValueError:
        print("Type a fraction like 1/7 or 3/17."); return
    if bottom < 2 or top < 1:
        print("Choose a top of at least 1 and a bottom of at least 2."); return
    whole = top // bottom
    digits, rems = long_division(top, bottom, bottom + 1)
    seen, start, stop = {}, None, None
    for i, r in enumerate(rems):
        if r == 0: stop = i; break
        if r in seen: start, stop = seen[r], i; break
        seen[r] = i
    if start is None:
        print(f"{top}/{bottom} = {whole}.{''.join(map(str, digits[:stop]))}   (the decimal ends: a remainder of 0)")
    else:
        pre, rep = "".join(map(str, digits[:start])), "".join(map(str, digits[start:stop]))
        print(f"{top}/{bottom} = {whole}.{pre}({rep}) repeating   ({len(rep)} digits repeat)")
    print("remainders:", ", ".join(map(str, rems[:stop + 1])))
    n = stop + 1
    fig, ax = plt.subplots(figsize=(min(12, 1 + 0.6 * n), 1.8))
    for i, r in enumerate(rems[:n]):
        col = CORAL if (start is not None and r == rems[start] and i in (start, stop)) else TEAL
        ax.add_patch(plt.Rectangle((i, 0), 0.9, 0.9, color=col, alpha=0.85))
        ax.text(i + 0.45, 0.45, str(r), ha="center", va="center", color=SLATE, fontsize=12, weight="bold")
    ax.set_xlim(-0.2, n + 0.1); ax.set_ylim(-0.1, 1.0); ax.axis("off"); ax.set_title("remainders, step by step")
    plt.show()

interact(remainders, fraction=Text(value="1/7", description="fraction", continuous_update=False));

## 2 · A repeating decimal back into a fraction

Type a decimal with the repeating part in brackets, like `0.(7)` for 0.777… or `0.1(6)` for 0.1666… The trick from
the video: multiply by 10 so many times that the repeats line up, then subtract, and the endless tail cancels.

In [ ]:
def to_fraction(s="0.(7)"):
    s = s.strip()
    if "(" not in s:
        print(f"{s} = {Fraction(s)}"); return
    head, rep = s.split("(")
    rep = rep.rstrip(")")
    whole, _, pre = head.partition(".")
    k, m = len(pre), len(rep)
    big = int(whole + pre + rep) - int(whole + pre or "0")        # 10^(k+m) x − 10^k x
    f = Fraction(big, 10 ** (k + m) - 10 ** k)
    print(f"x = {s}")
    print(f"10^{k + m} x − 10^{k} x = {big}    so    {10 ** (k + m) - 10 ** k} x = {big}")
    print(f"x = {big}/{10 ** (k + m) - 10 ** k} = {f}    (check: {float(f):.12f})")

interact(to_fraction, s=Text(value="0.(7)", description="decimal"));

## 3 · Hunt for a fraction whose square is 2

Choose the bottom number. The cell finds the best top number for it and squares the fraction. You'll get very
close. You'll never land on exactly 2: the even-and-odd argument in the video says why.

In [ ]:
def hunt(bottom=5):
    top = round(bottom * math.sqrt(2))
    f = Fraction(top, bottom)
    sq = f * f
    print(f"best fraction with bottom {bottom}: {top}/{bottom} = {float(f):.10f}")
    print(f"its square: {sq} = {float(sq):.10f}    (miss: {float(sq - 2):+.2e})")
    best = [abs(Fraction(round(b * math.sqrt(2)), b) ** 2 - 2) for b in range(1, bottom + 1)]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.semilogy(range(1, bottom + 1), [float(x) for x in best], color=TEAL, lw=1.2)
    ax.semilogy([bottom], [float(abs(sq - 2))], "o", color=CORAL, ms=8)
    ax.set_xlabel("bottom number"); ax.set_ylabel("how far the square is from 2"); plt.show()

interact(hunt, bottom=IntSlider(value=5, min=1, max=1000, description="bottom"));

## 4 · Multiplying by i is a quarter turn

Pick a starting number a + bi and how many times to multiply by i. Each multiplication turns the arrow a quarter
turn round zero, and its length never changes.

In [ ]:
def quarter_turns(a=3, b=2, times=1):
    z = complex(a, b)
    fig, ax = plt.subplots(figsize=(5, 5))
    for k in range(times + 1):
        w = z * 1j ** k
        col = CORAL if k == times else (TEAL if k == 0 else DIM)
        ax.annotate("", xy=(w.real, w.imag), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=col, lw=2))
        ax.text(w.real * 1.08, w.imag * 1.08, f"{w.real:g} {'+' if w.imag >= 0 else '−'} {abs(w.imag):g}i", color=col)
    r = abs(z) * 1.3 + 0.5
    ax.set_xlim(-r, r); ax.set_ylim(-r, r); ax.set_aspect("equal")
    ax.axhline(0, color=DIM, lw=1); ax.axvline(0, color=DIM, lw=1)
    ax.set_title(f"({a} + {b}i) × i^{times}"); plt.show()
    print(f"length before: {abs(z):.4f}   after: {abs(z * 1j ** times):.4f}")

interact(quarter_turns, a=IntSlider(value=3, min=-5, max=5), b=IntSlider(value=2, min=-5, max=5),
         times=IntSlider(value=1, min=0, max=4, description="× i, times"));

## 5 · Hamilton's quaternions: order matters

A quaternion is w + xi + yj + zk. Hamilton's rule, cut into a Dublin bridge in 1843: i² = j² = k² = ijk = −1.
Pick two of i, j, k and multiply them both ways round.

In [ ]:
def qmul(p, q):
    w1, x1, y1, z1 = p; w2, x2, y2, z2 = q
    return (w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2, w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
            w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2, w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2)

UNITS = {"1": (1, 0, 0, 0), "i": (0, 1, 0, 0), "j": (0, 0, 1, 0), "k": (0, 0, 0, 1)}

def name(q):
    for n, u in UNITS.items():
        if q == u: return n
        if q == tuple(-c for c in u): return "−" + n
    return str(q)

def order(first="i", second="j"):
    print(f"{first} × {second} = {name(qmul(UNITS[first], UNITS[second]))}")
    print(f"{second} × {first} = {name(qmul(UNITS[second], UNITS[first]))}")
    print("\nand for ordinary numbers: 3 × 5 =", 3 * 5, " 5 × 3 =", 5 * 3)

interact(order, first=Dropdown(options=["i", "j", "k"], value="i"), second=Dropdown(options=["i", "j", "k"], value="j"));

## 6 · The challenge from the video

Which fraction is 0.1666…, with the six repeating forever? Try it first (hint: you need both 10x and 100x), then
press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("0.1666… = ?   (press the button when you're ready)"); return
    print("x = 0.1666…   10x = 1.666…   100x = 16.666…")
    print("100x − 10x = 15, so 90x = 15 and x = 15/90 =", Fraction(15, 90))

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
Rhind papyrus problem 3 (British Museum); the Gwalior inscription of 876 (Chaturbhuj temple); Brahmagupta,
*Brāhmasphuṭasiddhānta* (628); Michael Stifel, *Arithmetica integra* (1544); Caspar Wessel (1797, printed 1799);
W. R. Hamilton, letter to his son, 5 August 1865 (Trinity College Dublin). All retrieved 2026-11-15. Every number
above is computed in the cells.